# 03 · Filter & Rank — run the shared multi-layer filter on the campaign pool

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 25** you build `fp.Design` objects from your campaign pool (worked
example: `design_type="binder"`; use YOUR chosen type), call `fp.run_pipeline(...)`, and
`fp.report(...)` the survival funnel + ranked CSV. This is the **classical single-cutoff filter** —
the baseline the ML success predictor (notebook 04) must beat (D3 part 1).

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back — and the capstone's whole point is that notebook 04 *proposes* such an
> improvement (a learned predictor) from cohort data.

Run `00`-`02` first so `results/campaign_designs.csv` exists.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. The
cutoffs depend on `design_type` (binder/enzyme/antibody/monomer).

In [ ]:
import filtering_pipeline as fp
import pandas as pd

DESIGN_TYPE = "binder"     # keep consistent with notebooks 01-02 (your chosen type)
print("Loaded shared filtering_pipeline from:", fp.__file__)
print(f"{DESIGN_TYPE} cutoffs:", fp.DEFAULT_CUTOFFS[DESIGN_TYPE])

## Build `fp.Design` objects from the campaign pool

Map each campaign row onto `fp.Design`. The metrics drive the layers: `scrmsd`/`plddt`/
`pae_interaction` (Layer 1 self-consistency), `rosetta_dG`/`shape_complementarity`/`solubility`
(Layer 3 physics). We regenerate the pool deterministically if a fresh session lost the CSV. (Mock has
no independent orthogonal predictor, so we run Layers 1+3; on Colab add a second predictor for
Layer 2.)

In [ ]:
import os
import campaign_tools as ct

if not os.path.exists("results/campaign_designs.csv"):
    HS = ct.parse_hotspots("A56,A66,A115")
    pool = ct.generate_designs("EXAMPLE_TARGET", HS, n=200, design_type=DESIGN_TYPE, tool="mock")
    ct.score_designs(pool, tool="mock")
    ct.pool_to_df(pool).to_csv("results/campaign_designs.csv", index=False)

camp = pd.read_csv("results/campaign_designs.csv")

def row_to_design(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type=DESIGN_TYPE,
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3), tm_to_pdb=r.get("tm_to_pdb"),
        catalytic_geom_rmsd=r.get("catalytic_geom_rmsd"),
        extra={"hotspot_overlap": r.get("hotspot_overlap"), "target": r.get("target")},
    )

designs = [row_to_design(r) for _, r in camp.iterrows()]
print(f"built {len(designs)} fp.Design objects (design_type={DESIGN_TYPE})")

## Run the pipeline + report (the classical single-cutoff filter)

`fp.run_pipeline(design_type=...)` applies the cutoffs in order and returns a ranked DataFrame with
survival counts in `df.attrs`. `fp.report(...)` prints the hit-rate accounting and draws the survival
funnel. Read the bars as a funnel: steep drops show which layer discriminates. **This survival/hit
rate is the BASELINE** that notebook 04's learned predictor is compared against.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

df_ranked = fp.run_pipeline(designs, design_type=DESIGN_TYPE, use_layers=(1, 3))
df_ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(df_ranked, top_n=15, save_prefix="results/p25")
n = df_ranked.attrs["n_total"]; passed = int((df_ranked["layers_passed"] >= 3).sum())
print(f"\nclassical filter hit rate (all layers): {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")
print("saved results/p25_survival.png + results/p25_ranked.csv")
top

## Honest hit-rate accounting

Report `N passing all layers / N generated`, with the layer-by-layer survival counts — survival is
*enrichment*, not *correctness*. This is the classical-filter number the capstone tries to improve on
by **learning** a better triage rule from cohort data (notebook 04). Mock numbers are SYNTHETIC.

In [ ]:
print("layers_passed distribution:", df_ranked["layers_passed"].value_counts().sort_index().to_dict())
print(f"all-layers survivors = {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")
print("\nRemember: a passing design is a HYPOTHESIS. The filter enriches; it does not guarantee.")

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module (`design_type=` your type), not a one-off script.
- [ ] Survival-at-each-layer reported (funnel figure `results/p25_survival.png`).
- [ ] Honest classical-filter hit-rate accounting (N pass / N generated) — the baseline for notebook 04.
- [ ] Mapping assumptions (which fields → which `Design` attributes) written down.

**Next:** `04_validate.ipynb` — train the ML success predictor on the cohort table and benchmark it
against these single-metric cutoffs.